# Modules




This notebook performs community detection, module rewiring summaries, and module eigengene estimation from the GCC exports produced in `network_and_gcc`.




## Inputs




- `RNA_CSD_GCC.txt` and `Protein_CSD_GCC.txt` from `network_and_gcc`


- Harmonized RNA and protein expression matrices for healthy and cancer samples




## Outputs




- RNA and protein Leiden module folders with module and edge tables


- RNA and protein rewiring summary files


- RNA and protein module eigengene matrices for downstream survival analysis




## Pipeline




1. Load the RNA and protein GCC exports and build graph objects for each modality.


2. Detect Leiden modules and optionally project them into Cytoscape.


3. Compute module rewiring summaries from the exported module edge tables.


4. Estimate module eigengenes for healthy and cancer samples and save them for downstream analyses.




## Setup




The first code cell imports notebook-specific helpers from `Pipeline/4_Network_Analysis/utils/modules_and_eigengenes.py`, uses shared context helpers from `Pipeline/4_Network_Analysis/utils/network_and_gcc.py`, initializes `AnalysisContext`, and loads the GCC inputs reused across the rest of the notebook.

In [4]:
import importlib

import pandas as pd
from IPython.display import display

import utils as notebook_utils

notebook_utils = importlib.reload(notebook_utils)

from utils import (
    build_analysis_context,
    compute_module_eigengenes,
    compute_rewiring_scores,
    convert_expression_index_to_symbols,
    ensure_gcc_output,
    leiden_modules_with_edge_types,
    load_edges_clean,
    load_expression_matrix,
    load_graph,
 )

ctx = build_analysis_context(run_cytoscape=True, run_enrichment=False, save_figures=True)
rna_gcc_path = ensure_gcc_output(
    ctx.network_files['rna_csd'],
    ctx.output_path('RNA_CSD_GCC.txt'),
)
prot_gcc_path = ensure_gcc_output(
    ctx.network_files['prot_csd'],
    ctx.output_path('Protein_CSD_GCC.txt'),
)

gcc_rna = load_edges_clean(pd.read_csv(rna_gcc_path, sep='\t'))
gcc_prot = load_edges_clean(pd.read_csv(prot_gcc_path, sep='\t'))
edges_rna_gcc = gcc_rna.copy()
edges_prot_gcc = gcc_prot.copy()
G_rna_gcc = load_graph(gcc_rna)
G_prot_gcc = load_graph(gcc_prot)

CYTOSCAPE_MIN_MODULE_SIZE = 1


Using cached GCC export: C:\Users\tiril\Master-Tiril\results\notebooks\03_modules\RNA_CSD_GCC.txt
Using cached GCC export: C:\Users\tiril\Master-Tiril\results\notebooks\03_modules\Protein_CSD_GCC.txt


## Step 1: Community detection outputs

This step runs or reloads the RNA and protein Leiden module tables and summarizes the module counts that are carried into the downstream analyses.

In [3]:
LEIDEN_CONFIGS = [
    {
        'label': 'RNA',
        'graph': G_rna_gcc,
        'edges': edges_rna_gcc,
        'resolution': 1.0,
        'output_dir': 'RNA_Leiden_Modules_1.0',
    },
    {
        'label': 'Protein',
        'graph': G_prot_gcc,
        'edges': edges_prot_gcc,
        'resolution': 1.0,
        'output_dir': 'Protein_Leiden_Modules_1.0',
    },
]

LEIDEN_RECOMPUTE = False

leiden_run_summary = []
for config in LEIDEN_CONFIGS:
    output_dir = ctx.output_path(config['output_dir'])
    module_table_path = output_dir / 'module_table.csv'

    if LEIDEN_RECOMPUTE or not module_table_path.exists():
        output_dir.mkdir(parents=True, exist_ok=True)
        leiden_modules_with_edge_types(
            config['graph'],
            config['edges'],
            resolution=config['resolution'],
            output_dir=output_dir,
        )

    module_table = pd.read_csv(module_table_path)
    leiden_run_summary.append(
        {
            'dataset': config['label'],
            'resolution': config['resolution'],
            'n_modules': module_table['Module'].nunique(),
            'n_genes': module_table['Gene'].nunique(),
            'module_table': str(module_table_path),
        }
    )

display(pd.DataFrame(leiden_run_summary))

# check how many modules have at least 20 genes
for config in LEIDEN_CONFIGS:
    module_table = pd.read_csv(ctx.output_path(config['output_dir']) / 'module_table.csv')
    module_counts = module_table['Module'].value_counts()
    n_modules_20_genes = (module_counts >= 20).sum()
    print(f"{config['label']} - Modules with >= 20 genes: {n_modules_20_genes} / {len(module_counts)}")

,dataset,resolution,n_modules,n_genes,module_table
0,RNA,1.0,8,1314,C:\Users\tiril\Master-Tiril\results\notebooks\...
1,Protein,1.0,6,1034,C:\Users\tiril\Master-Tiril\results\notebooks\...


RNA - Modules with >= 20 genes: 7 / 8
Protein - Modules with >= 20 genes: 6 / 6


In [ ]:


if ctx.run_cytoscape:
    modules_rna_table = pd.read_csv(ctx.output_path('RNA_Leiden_Modules_1.0', 'module_table.csv'))
    modules_prot_table = pd.read_csv(ctx.output_path('Protein_Leiden_Modules_1.0', 'module_table.csv'))

    rna_module_coloring = notebook_utils.mark_modules_in_cytoscape(
        'RNA_CSD_GCC', # lag en kopi i cytoscape
        modules_rna_table,
        min_module_size=CYTOSCAPE_MIN_MODULE_SIZE,
    )
    prot_module_coloring = notebook_utils.mark_modules_in_cytoscape(
        'Protein_CSD_GCC', # lag en kopi i cytoscape
        modules_prot_table,
        min_module_size=CYTOSCAPE_MIN_MODULE_SIZE,
    )

    rna_module_subnetworks = notebook_utils.create_module_subnetworks_in_cytoscape(
        'RNA_CSD_GCC',
        modules_rna_table,
        min_module_size=CYTOSCAPE_MIN_MODULE_SIZE,
        subnetwork_prefix='RNA_CSD_GCC',
    )
    prot_module_subnetworks = notebook_utils.create_module_subnetworks_in_cytoscape(
        'Protein_CSD_GCC',
        modules_prot_table,
        min_module_size=CYTOSCAPE_MIN_MODULE_SIZE,
        subnetwork_prefix='Protein_CSD_GCC',
    )

    display(rna_module_coloring)
    display(prot_module_coloring)
    display(rna_module_subnetworks)
    display(prot_module_subnetworks)
else:
    print('Skipping Cytoscape module coloring and subnetwork creation: ctx.run_cytoscape=False')

You are connected to Cytoscape!


## Step 2: Module rewiring scores

This step uses `compute_rewiring_scores` on the saved module edge tables and writes per-module rewiring summaries for downstream ranking.

In [4]:

prot_module_edge_table = pd.read_csv(ctx.output_path('Protein_Leiden_Modules_1.0', 'module_edge_table.tsv'), sep='	')


prot_rewiring_scores = compute_rewiring_scores(prot_module_edge_table, output_file=ctx.output_path('Protein_Leiden_Modules_1.0', 'rewiring_scores.csv'))

rna_module_edge_table = pd.read_csv(ctx.output_path('RNA_Leiden_Modules_1.0', 'module_edge_table.tsv'), sep='	')


rna_rewiring_scores = compute_rewiring_scores(rna_module_edge_table, output_file=ctx.output_path('RNA_Leiden_Modules_1.0', 'rewiring_scores.csv'))


Saved rewiring scores for 6 modules to C:\Users\tiril\Master-Tiril\results\notebooks\03_modules\Protein_Leiden_Modules_1.0\rewiring_scores.csv
Saved rewiring scores for 8 modules to C:\Users\tiril\Master-Tiril\results\notebooks\03_modules\RNA_Leiden_Modules_1.0\rewiring_scores.csv


## Step 3: Module eigengenes

This step builds expression matrices and computes module eigengenes for healthy and cancer samples.

In [5]:
expr_rna_healthy = convert_expression_index_to_symbols(load_expression_matrix(ctx.expression_files['rna_healthy']))
expr_rna_cancer = convert_expression_index_to_symbols(load_expression_matrix(ctx.expression_files['rna_cancer']))
expr_prot_healthy = convert_expression_index_to_symbols(load_expression_matrix(ctx.expression_files['prot_healthy']))
expr_prot_cancer = convert_expression_index_to_symbols(load_expression_matrix(ctx.expression_files['prot_cancer']))

rna_expression_symbols_path = ctx.output_dir / 'RNA_cancer_expression_symbols.csv'
protein_expression_symbols_path = ctx.output_dir / 'Protein_cancer_expression_symbols.csv'
rna_module_eigengenes_path = ctx.output_dir / 'RNA_module_eigengenes.csv'
protein_module_eigengenes_path = ctx.output_dir / 'Protein_module_eigengenes.csv'

expr_rna_cancer.to_csv(rna_expression_symbols_path)
expr_prot_cancer.to_csv(protein_expression_symbols_path)

eigengenes_rna_healthy, eigengenes_rna_cancer = compute_module_eigengenes(
    expr_rna_healthy,
    expr_rna_cancer,
    pd.read_csv(ctx.output_path('RNA_Leiden_Modules_1.0', 'module_table.csv')),
)
eigengenes_prot_healthy, eigengenes_prot_cancer = compute_module_eigengenes(
    expr_prot_healthy,
    expr_prot_cancer,
    pd.read_csv(ctx.output_path('Protein_Leiden_Modules_1.0', 'module_table.csv')),
)

eigengenes_rna = pd.concat([
    eigengenes_rna_healthy.assign(condition='Healthy'),
    eigengenes_rna_cancer.assign(condition='Cancer'),
])
eigengenes_prot = pd.concat([
    eigengenes_prot_healthy.assign(condition='Healthy'),
    eigengenes_prot_cancer.assign(condition='Cancer'),
])

eigengenes_rna.to_csv(rna_module_eigengenes_path)
eigengenes_prot.to_csv(protein_module_eigengenes_path)

display(eigengenes_rna.head())
display(eigengenes_prot.head())

,1,2,3,4,5,6,7,8,condition
C3L-02610,-15.136766,-3.551596,18.376570,-9.876694,-14.348271,-14.072649,-6.035906,-2.958041,Healthy
C3N-01998,-17.613826,-16.182196,-4.123588,-9.318731,-1.341156,-9.578854,2.748325,-1.634467,Healthy
C3L-01637,-15.360981,-6.724977,14.112906,-10.848061,-10.860155,-13.468577,-3.737822,-3.733378,Healthy
C3L-02606,-17.229877,-9.087789,12.820583,-9.313042,-8.106416,-13.791621,-0.841325,-3.051866,Healthy
C3N-00436,-28.449931,-16.143497,9.156438,-7.843866,-6.801414,-18.485780,0.261830,-2.359287,Healthy


,1,2,3,4,5,6,condition
C3L-03123,-9.271875,15.466931,8.093375,-11.446633,8.362843,-2.566289,Healthy
C3N-03884,-10.527052,13.724055,5.876438,-12.610119,10.028616,-2.301724,Healthy
C3L-03630,-13.214984,26.233616,17.927602,-11.872231,12.875626,-3.016626,Healthy
C3L-03356,-9.376190,13.226169,7.479232,-9.947214,8.785531,-2.077696,Healthy
C3L-01054,-5.120371,-1.786826,-3.543703,-2.278814,-1.139139,-1.808621,Healthy
